# Phase 8 — Define RUL before creating labels

## Start with the definition

Remaining Useful Life (RUL) is the remaining operating duration, cycles, or accumulated dose from a current observation until a **predefined, physically meaningful End of Life (EoL)**:

$$\mathrm{RUL}(t) = t_{\mathrm{EoL}} - t$$

Both times must use the same clock and unit, and `t` must represent elapsed exposure from a defensible Beginning of Life (BoL). For cycling systems, the more appropriate form could be cycles-to-EoL; for an accelerated aging test, it could be time under a specified stress or cumulative thermal/electrical dose. Calendar time is not automatically useful life.

**This phase defines what the available records support. It does not create RUL labels.** A last measurement is an observation boundary, not an observed failure. If a device has not failed when observation stops, its true RUL is unknown and the record is right-censored (assuming it was confirmed operational at that point).

## 1. BoL: what can count as the beginning?

A useful BoL is the start of the defined use/aging exposure, with a known device condition and a recorded reference measurement. The device aging log for this square-gate experiment has `begin` entries for Devices 2–5, so it can identify a **test-start event**. It does not establish a standardized healthy-condition baseline, the exact instant of first damaging exposure for every capture, or a cycle count. The first transient timestamp is simply the first available transient observation; it is not automatically BoL.

For a future study, record the device's initial qualification/health state, stress start time, actual stress cycles and interruptions, and synchronized timestamps. Until then, if analysis uses first observation as time zero, label it explicitly as `first_observation_time` (a coordinate origin), not a measured healthy BoL.

## 2. EoL and failure: what is actually recorded?

The aging log is a human-readable experiment record, not a uniform failure-label table. Its notes for the selected Devices 2–5 square-gate aging experiment include:

| Device | Log evidence | What it supports—and does not support |
|---|---|---|
| Device 2 | Test begins, conditions are stepped, a continuation segment is recorded, and a later low-temperature `check` file exists. No explicit failure event is listed in the log excerpt. | No verified EoL time; ending the main segment or having a later check is not failure. |
| Device 3 | Log records the temperature response becoming sluggish, then notes `end experiment…device latchup` at about 04:02. | An explicit abnormal event is recorded and may be a failure candidate. No standardized failure criterion, independent confirmation, or trustworthy aligned EoL timestamp is supplied in a machine-readable label. |
| Device 4 | Test and continuation segment with stress settings are logged; no explicit failure event is listed. | No verified EoL time. The last file/capture is an observation endpoint only. |
| Device 5 | Log records `end` at about 12:54, followed by the note that the device is no longer able to draw current. | Strong evidence of an operational failure-like event, but not a standardized criterion or a precisely synchronized failure timestamp. |

The archive's other experiments mention latch-up/thermal runaway, but those belong to different experiment folders and must not be imported as labels for these four devices. Failure criteria should be specified before testing (for example, inability to draw a required current under a defined test, a confirmed latch-up event, or exceeding a specified electrical limit). Those criteria are not provided uniformly for this selected experiment.

## 3. Inspect available capture times and check their consistency

The Phase 5 feature table carries both a text `date` captured inside each MAT record and a numeric `timeEpoch`. Before using either to calculate time-to-event, inspect them and compare to the aging-log clock times. Source-file dates and timestamps describe observations; they do not themselves denote failure. A mismatch between clocks must be resolved before aligning a log event to a capture.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "data" / "features").is_dir():
    project_root = project_root.parent

feature_path = project_root / "data" / "features" / "transient_candidate_features.csv"
if not feature_path.is_file():
    raise FileNotFoundError(
        f"{feature_path} was not found. Run notebooks/03_feature_engineering.ipynb first."
    )

captures = pd.read_csv(feature_path)
required_columns = {"device_folder", "source_file", "date", "timeEpoch"}
missing = sorted(required_columns.difference(captures.columns))
if missing:
    raise ValueError(f"Feature table is missing required columns: {missing}")

captures["timeEpoch"] = pd.to_numeric(captures["timeEpoch"], errors="raise")
captures["is_check_file"] = captures["source_file"].str.contains("check", case=False, regex=False)
captures["date_parsed"] = pd.to_datetime(
    captures["date"], format="%m/%d/%Y %I:%M:%S %p.%f", errors="coerce"
)
aging_captures = captures.loc[~captures["is_check_file"]].copy()

def describe_time(group):
    sorted_group = group.sort_values("timeEpoch")
    return pd.Series({
        "captures": len(group),
        "first_source_date_text": sorted_group["date"].iloc[0],
        "last_source_date_text": sorted_group["date"].iloc[-1],
        "first_timeEpoch_UTC": pd.to_datetime(sorted_group["timeEpoch"].iloc[0], unit="s", utc=True),
        "last_timeEpoch_UTC": pd.to_datetime(sorted_group["timeEpoch"].iloc[-1], unit="s", utc=True),
        "all_source_dates_parse": bool(group["date_parsed"].notna().all()),
    })

time_inventory = pd.DataFrame({
    device: describe_time(group)
    for device, group in aging_captures.groupby("device_folder", observed=True, sort=True)
}).T
time_inventory

,captures,first_source_date_text,last_source_date_text,first_timeEpoch_UTC,last_timeEpoch_UTC,all_source_dates_parse
Device 2,175,12/24/2008 9:59:47 AM.786743640,12/24/2008 11:08:58 AM.161743640,2074-12-25 17:59:47+00:00,2074-12-25 19:08:58+00:00,True
Device 3,152,12/24/2008 3:03:56 PM.416341781,12/24/2008 4:02:15 PM.783341884,2074-12-25 23:03:56+00:00,2074-12-26 00:02:15+00:00,True
Device 4,255,12/24/2008 4:15:19 PM.065341949,12/24/2008 6:06:05 PM.117341995,2074-12-26 00:15:19+00:00,2074-12-26 02:06:05+00:00,True
Device 5,106,12/24/2008 12:13:54 PM.168751239,12/24/2008 12:52:44 PM.985751152,2074-12-25 20:13:54+00:00,2074-12-25 20:52:44+00:00,True


### Time-coordinate warning

In this extracted table, the MAT `date` text is in December 2008, while interpreting `timeEpoch` as Unix seconds produces dates in December 2074. This large disagreement is not a timezone adjustment. **Do not use the numeric `timeEpoch` to align aging-log events or calculate EoL/RUL until the timestamp encoding is investigated and reconciled.** A log time-of-day is also not sufficient by itself if the date, timezone, device identity, and capture clock synchronization are unknown.

This is an additional reason the current records cannot support reliable, numeric RUL labels even for Devices 3 and 5, whose logs contain failure-like event notes.

## 4. Dataset-specific decisions

1. **BoL?** The log gives approximate test `begin` events, but no uniform verified healthy baseline or complete stress-dose/cycle history. First capture can be an explicit *observation-origin assumption*, not confirmed BoL.
2. **EoL?** There is no standardized, machine-readable EoL field for all four devices. Devices 3 (latch-up note) and 5 (cannot draw current note) have event evidence; Devices 2 and 4 have no explicit failure in these log entries.
3. **Failure definition?** No common predeclared threshold/test criterion is supplied for this cohort. The Device 3/5 notes are failure-like observations, not a uniform engineering EoL rule.
4. **Is failure explicitly recorded?** Partly: human log notes for Devices 3 and 5. Not as a synchronized, consistently coded failure label for every device. Devices 2 and 4 should be treated as having unknown EoL, not assumed failures at their last observations.
5. **Can RUL be defined reliably?** Not as a cohort-wide, measured target from the current dataset. Missing/ambiguous BoL, no uniform EoL, incomplete event alignment, and the `date`/`timeEpoch` mismatch prevent a reliable time-to-failure label.
6. **If EoL is estimated?** It must be stored as an **assumed/estimated EoL**, with its rule, evidence, and uncertainty. Any derived value must be called `assumed_RUL` or `time_to_assumed_EoL`, never measured failure RUL. Do not make that assumption silently.

## 5. Adapt the RUL equation to the evidence

If a verified EoL time and current time share a validated coordinate, then:

$$\mathrm{RUL}_{\mathrm{measured}}(t) = t_{\mathrm{verified\ EoL}} - t_{\mathrm{current}}$$

Use elapsed exposure since BoL (or cycles/dose) when that is the intended useful-life measure. For this dataset, a conditional calculation would be:

$$\mathrm{time\ to\ assumed\ EoL}(t) = t_{\mathrm{assumed\ EoL}} - t_{\mathrm{observation}}$$

but only after stating the assumed event and resolving timestamp alignment. It is not a measured-failure RUL. For a run that ends without an observed failure, we only know that it was observed through its last valid time; if confirmed still functioning then, the failure time is **right-censored** beyond that time, and exact RUL is unknown. If operational status at the last record is also unknown, even that censoring statement requires qualification.

The existing prototype's `remaining_observed_hours = last_record_time - current_record_time` is therefore a **remaining observation-horizon proxy**, not RUL. It sets every run's endpoint to data collection stopping, even when no failure was recorded. Do not interpret its labels or model metrics as physical remaining life.

## Phase 8 conclusion and learning check

**Decision:** do not create measured RUL labels from this dataset yet. The most defensible next step is to verify timestamp encoding/synchronization, review the original experimental protocol and device log with domain experts, establish a uniform operational failure criterion, and identify whether each device was known to be operating at its last observation. Record event times and stress exposure in a validated per-device table. Only then decide whether Devices 3/5 can supply event-time labels and whether Devices 2/4 are censored observations.

1. Why is the first available capture not automatically BoL?
2. Which devices have explicit failure-like notes, and why are those not automatically precise EoL labels?
3. What does the `date` versus `timeEpoch` discrepancy imply for RUL calculations?
4. When is a device right-censored, and what must be known at its last observation?
5. How does remaining time to the last record differ from time to physical EoL?
6. If you proposed an assumed EoL rule, what metadata and label naming would keep it distinct from measured failure?

No labels, EoL estimates, assumptions, or model targets are generated by this notebook.